In [ ]:
import threading
from transformers import AutoTokenizer,AutoModelForCausalLM, TextIteratorStreamer
from time import time
from tokenize import tokenize

tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2')

prompt = "The Next day is Bright"
tokens  = tokenizer.encode(prompt, return_tensors = "pt")


def stream_output(use_cache):
    streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)
    thread = threading.Thread(target=model.generate, kwargs={
        "input_ids" : tokens,
        "max_new_tokens": 100,
        "use_cache": use_cache,
        "streamer":streamer           
    })

    thread.start()

    start_time = time()
    for token in streamer:
        print(token, end = "", flush=True)

    end_time = time()
    thread.join()

    elapsed_time = end_time - start_time
stream_output(use_cache=True)
stream_output(use_cache=False)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11234.81it/s]


The Next day is Bright's birthday, and he's going to be in the hospital with a broken leg. He's going to be in the hospital with a broken leg. He's going to be in the hospital with a broken leg. He's going to be in the hospital with a broken leg. He's going to be in the hospital with a broken leg. He's going to be in the hospital with a broken leg. He's going to be in the hospital with a broken leg. He's going to beThe Next day is Bright's birthday, and he's going to be in the hospital with a broken leg. He's going to be in the hospital with a broken leg. He's going to be in the hospital with a broken leg. He's going to be in the hospital with a broken leg. He's going to be in the hospital with a broken leg. He's going to be in the hospital with a broken leg. He's going to be in the hospital with a broken leg. He's going to be

In [7]:
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

# --- Timing without KV cache ---
print("Generating without KV Cache...")
start_time_without_cache = time()
output_without_cache = model.generate(
    input_ids,
    max_new_tokens=100,
    use_cache=False, # Explicitly disable the cache
    attention_mask=attention_mask
)
end_time_without_cache = time()
duration_without_cache = end_time_without_cache - start_time_without_cache
print(f"Time without KV Cache: {duration_without_cache:.4f} seconds\n")


# --- Timing with KV cache ---
print("Generating with KV Cache...")
start_time_with_cache = time()
output_with_cache = model.generate(
    input_ids,
    max_new_tokens=100,
    use_cache=True, # Explicitly enable the cache
    attention_mask=attention_mask
)
end_time_with_cache = time()
duration_with_cache = end_time_with_cache - start_time_with_cache
print(f"Time with KV Cache: {duration_with_cache:.4f} seconds\n")


# --- Calculate and print the speedup ---
speedup = duration_without_cache / duration_with_cache
print(f"KV Cache Speedup: {speedup:.2f}x")
     

Generating without KV Cache...
Time without KV Cache: 7.8280 seconds

Generating with KV Cache...
Time with KV Cache: 4.6588 seconds

KV Cache Speedup: 1.68x
